# Project 8 – Train the plane detector (OSS4008A)
**Notebook 04:** fine-tunes a pretrained object-detection model to find planes in the Ontario aerial image chips labelled with LabelMe.

**The idea in plain words**
- We start from **Faster R-CNN**, a well-known detector that has already learned from millions of everyday photos (COCO dataset) how to find objects and draw boxes around them.
- We **fine-tune** it: keep what it knows about shapes and edges, and teach only its last layer one new class, **plane**, from our own labelled chips.
- Aerial images have no "up", so every training image is also shown **rotated and mirrored**: 8 versions of each chip for free.
- We hold back **20% of the chips** that the model never sees during training, and measure on those how many planes it finds (**recall**), how many of its boxes are real planes (**precision**), and the overall score (**AP50**).

**Data:** chips from notebook 03 (Ontario aerial imagery, Open Government Licence – Ontario) + labels drawn in LabelMe (`plane_labels.zip`).

**How to run:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all. Cell 3 asks you to upload `plane_labels.zip`.

In [ ]:
# Cell 1 – Libraries and GPU check
import os, io, glob, json, random, zipfile, time
import numpy as np
from PIL import Image
import torch, torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import matplotlib.pyplot as plt
import matplotlib.patches as patches

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| device:', device)
if device.type == 'cpu':
    print('⚠️ No GPU: Runtime → Change runtime type → T4 GPU, then run again (CPU training is very slow).')

random.seed(42); np.random.seed(42); torch.manual_seed(42)   # same split and results every run

In [ ]:
# Cell 2 – Folders in Google Drive (the chips from notebook 03 are already there)
from google.colab import drive, files
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/oss4008a_planes'
IMG_DIR = f'{DATA}/images'
ANN_DIR = f'{DATA}/annotations'                 # LabelMe .json files go here
RES_DIR = f'{DATA}/results'
MODEL_PATH = f'{DATA}/plane_detector.pt'        # the trained model (too big for GitHub: kept in Drive)
os.makedirs(ANN_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)
print('chips in Drive:', len(glob.glob(f'{IMG_DIR}/*.jpg')))

In [ ]:
# Cell 3 – Upload the LabelMe labels (plane_labels.zip from your Desktop), only needed the first time
if not glob.glob(f'{ANN_DIR}/*.json'):
    uploaded = files.upload()                                   # pick plane_labels.zip
    for name, data in uploaded.items():
        zipfile.ZipFile(io.BytesIO(data)).extractall(ANN_DIR)
print('label files:', len(glob.glob(f'{ANN_DIR}/*.json')))

## Helper functions
Reading LabelMe boxes, rotating/mirroring images with their boxes, and scoring predictions against the hand-drawn boxes.

In [ ]:
# Cell 4 – Helper functions
import json
import numpy as np


def load_boxes(json_path, label="plane"):
    """Read one LabelMe file and return its plane boxes as [x_min, y_min, x_max, y_max] in pixels.
    Works for rectangles and for polygons (e.g. from AI-assisted labelling): the box is the min/max of the points."""
    with open(json_path) as f:
        ann = json.load(f)
    boxes = []
    for shape in ann.get("shapes", []):
        if shape.get("label") != label or not shape.get("points"):
            continue
        pts = np.array(shape["points"], dtype=float)
        x0, y0 = pts.min(axis=0)
        x1, y1 = pts.max(axis=0)
        if x1 - x0 >= 2 and y1 - y0 >= 2:          # ignore accidental tiny boxes
            boxes.append([x0, y0, x1, y1])
    return boxes


def augment(img, boxes, k, flip_h, flip_v):
    """Rotate the image by k x 90 degrees and optionally flip it, moving the boxes the same way.
    Aerial images have no 'up', so rotated and flipped copies are just as realistic: free extra training data.
    img: H x W x 3 array (square chips); boxes: N x 4 array [x0, y0, x1, y1]."""
    img = np.asarray(img)
    b = np.asarray(boxes, dtype=float).reshape(-1, 4).copy()
    size = img.shape[1]                              # chips are square
    for _ in range(k % 4):                           # one 90-degree turn anticlockwise at a time
        img = np.rot90(img)
        # a point (x, y) moves to (y, size - x)
        b = np.stack([b[:, 1], size - b[:, 2], b[:, 3], size - b[:, 0]], axis=1)
    if flip_h:                                       # mirror left <-> right
        img = img[:, ::-1]
        b = np.stack([size - b[:, 2], b[:, 1], size - b[:, 0], b[:, 3]], axis=1)
    if flip_v:                                       # mirror top <-> bottom
        img = img[::-1, :]
        b = np.stack([b[:, 0], size - b[:, 3], b[:, 2], size - b[:, 1]], axis=1)
    return np.ascontiguousarray(img), b


def iou(a, b):
    """Intersection-over-union of two boxes [x0, y0, x1, y1]: 1 = identical, 0 = no overlap."""
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def evaluate(predictions, truths, iou_thr=0.5, score_thr=0.5):
    """Compare predicted boxes with the hand-drawn ones over a set of test images.
    predictions: list (one per image) of (boxes, scores); truths: list of box lists.
    Returns precision, recall, F1 at score_thr, AP at IoU 0.5, and the counting error."""
    # 1) Precision / recall / F1 at one confidence threshold
    tp = fp = fn = 0
    count_err = []
    for (pb, ps), tb in zip(predictions, truths):
        keep = [b for b, s in zip(pb, ps) if s >= score_thr]
        matched = set()
        for p in keep:
            best, best_j = 0.0, -1
            for j, t in enumerate(tb):
                if j not in matched and iou(p, t) > best:
                    best, best_j = iou(p, t), j
            if best >= iou_thr:
                tp += 1
                matched.add(best_j)
            else:
                fp += 1
        fn += len(tb) - len(matched)
        count_err.append(abs(len(keep) - len(tb)))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    # 2) Average precision (area under the precision-recall curve, all confidence levels)
    dets = []                                        # (score, image index, box)
    for i, (pb, ps) in enumerate(predictions):
        dets += [(s, i, b) for b, s in zip(pb, ps)]
    dets.sort(key=lambda d: -d[0])
    n_true = sum(len(t) for t in truths)
    used = [set() for _ in truths]
    tps, fps = [], []
    for s, i, b in dets:
        best, best_j = 0.0, -1
        for j, t in enumerate(truths[i]):
            if j not in used[i] and iou(b, t) > best:
                best, best_j = iou(b, t), j
        if best >= iou_thr:
            used[i].add(best_j); tps.append(1); fps.append(0)
        else:
            tps.append(0); fps.append(1)
    ap = 0.0
    if n_true and dets:
        tp_c, fp_c = np.cumsum(tps), np.cumsum(fps)
        rec = tp_c / n_true
        prec = tp_c / np.maximum(tp_c + fp_c, 1)
        # standard all-point interpolation (as in PASCAL VOC)
        mrec = np.concatenate([[0], rec, [1]])
        mpre = np.concatenate([[0], prec, [0]])
        for k in range(len(mpre) - 2, -1, -1):
            mpre[k] = max(mpre[k], mpre[k + 1])
        idx = np.where(mrec[1:] != mrec[:-1])[0]
        ap = float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
    return {"precision": precision, "recall": recall, "f1": f1, "AP50": ap,
            "true_planes": n_true, "TP": tp, "FP": fp, "FN": fn,
            "mean_count_error": float(np.mean(count_err)) if count_err else 0.0}

In [ ]:
# Cell 5 – Build the dataset: every chip with its boxes (chips without a label file = checked, no planes)
items = []                                       # (image path, list of boxes)
for img_path in sorted(glob.glob(f'{IMG_DIR}/*.jpg')):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    ann = f'{ANN_DIR}/{stem}.json'
    items.append((img_path, load_boxes(ann) if os.path.exists(ann) else []))

positives = [it for it in items if it[1]]
negatives = [it for it in items if not it[1]]
print(f'{len(items)} chips: {len(positives)} with planes ({sum(len(b) for _, b in positives)} planes), {len(negatives)} without')

# Keep some empty chips as "no plane here" examples (aprons with trucks, jet bridges, buildings),
# but not all of them, so the model does not mostly see empty images
random.shuffle(negatives)
negatives = negatives[:int(1.5 * len(positives))]

# 80 / 20 split, separately for chips with and without planes, so both sets get some of each
def split(lst, frac=0.2):
    lst = lst[:]; random.shuffle(lst)
    n_test = max(1, round(frac * len(lst)))
    return lst[n_test:], lst[:n_test]
pos_train, pos_test = split(positives)
neg_train, neg_test = split(negatives)
train_items, test_items = pos_train + neg_train, pos_test + neg_test
print(f'train: {len(train_items)} chips ({sum(len(b) for _, b in train_items)} planes) | '
      f'test: {len(test_items)} chips ({sum(len(b) for _, b in test_items)} planes)')

# Planes per airport (file names start with the airport code)
per_airport = {}
for p, b in positives:
    code = os.path.basename(p)[:3]
    per_airport[code] = per_airport.get(code, 0) + len(b)
print('planes per airport:', per_airport)

In [ ]:
# Cell 6 – PyTorch dataset: loads a chip, applies a random rotation/mirror when training
class PlaneDataset(torch.utils.data.Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, boxes = self.items[i]
        img = np.array(Image.open(path).convert('RGB'))
        b = np.array(boxes, dtype=np.float32).reshape(-1, 4)
        if self.train:                                   # random 0/90/180/270 turn + random mirrors
            img, b = augment(img, b, random.randint(0, 3), random.random() < 0.5, random.random() < 0.5)
        img_t = torch.from_numpy(img.copy()).permute(2, 0, 1).float() / 255.0   # H x W x 3 -> 3 x H x W, 0..1
        target = {'boxes': torch.as_tensor(b, dtype=torch.float32).reshape(-1, 4),
                  'labels': torch.ones((len(b),), dtype=torch.int64)}           # class 1 = plane
        return img_t, target

def collate(batch):
    return tuple(zip(*batch))                        # detection models take lists of images, not one stacked tensor
train_loader = torch.utils.data.DataLoader(PlaneDataset(train_items, train=True), batch_size=2,
                                           shuffle=True, num_workers=2, collate_fn=collate)

# Sanity check: show one augmented training chip with its boxes
img_t, tgt = PlaneDataset(pos_train, train=True)[0]
fig, ax = plt.subplots(figsize=(5, 5)); ax.imshow(img_t.permute(1, 2, 0).numpy())
for x0, y0, x1, y1 in tgt['boxes'].tolist():
    ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='lime', linewidth=2))
ax.set_title('training example (rotated/mirrored) with its boxes'); ax.axis('off'); plt.show()

In [ ]:
# Cell 7 – The model: Faster R-CNN pretrained on COCO, with a new last layer for 2 classes (background, plane)
model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
                                   min_size=1024, max_size=1024)   # keep chips at full size: small planes stay visible
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
model.to(device)
print('model ready')

In [ ]:
# Cell 8 – Fine-tune (about 5-10 minutes on a T4 GPU)
EPOCHS = 20
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=12, gamma=0.1)   # smaller steps for the last epochs

history = []
start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for imgs, targets in train_loader:
        imgs = [im.to(device) for im in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss = sum(model(imgs, targets).values())        # the model returns its losses while training
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    scheduler.step()
    history.append(total / len(train_loader))
    print(f'epoch {epoch:2d}/{EPOCHS}  loss {history[-1]:.3f}  ({time.time() - start:.0f} s)')

torch.save(model.state_dict(), MODEL_PATH)
print('\nSaved model to', MODEL_PATH)
plt.plot(range(1, EPOCHS + 1), history, marker='o'); plt.xlabel('epoch'); plt.ylabel('training loss')
plt.title('Training loss (lower = better fit)'); plt.savefig(f'{RES_DIR}/training_loss.png', dpi=120); plt.show()

In [ ]:
# Cell 9 – Test on the 20% of chips the model never saw
model.eval()
predictions, truths = [], []
with torch.no_grad():
    for path, boxes in test_items:
        img_t = torch.from_numpy(np.array(Image.open(path).convert('RGB'))).permute(2, 0, 1).float() / 255.0
        out = model([img_t.to(device)])[0]
        predictions.append((out['boxes'].cpu().tolist(), out['scores'].cpu().tolist()))
        truths.append(boxes)

metrics = evaluate(predictions, truths, iou_thr=0.5, score_thr=0.5)
for k, v in metrics.items():
    print(f'{k:>17}: {v:.3f}' if isinstance(v, float) else f'{k:>17}: {v}')
json.dump({'model': 'fasterrcnn_resnet50_fpn_v2 (COCO-pretrained, fine-tuned)', 'epochs': EPOCHS,
           'train_chips': len(train_items), 'test_chips': len(test_items), 'score_threshold': 0.5,
           'iou_threshold': 0.5, **metrics}, open(f'{RES_DIR}/test_metrics.json', 'w'), indent=2)

In [ ]:
# Cell 10 – Picture of the results: green = hand-drawn boxes, red = model (confidence ≥ 0.5)
show = [i for i, t in enumerate(truths) if t][:4] or list(range(min(4, len(truths))))
fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 5))
for ax, i in zip(np.atleast_1d(axes), show):
    ax.imshow(Image.open(test_items[i][0]))
    for x0, y0, x1, y1 in truths[i]:
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='lime', linewidth=2))
    for (x0, y0, x1, y1), s in zip(*predictions[i]):
        if s >= 0.5:
            ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='red', linewidth=1.5))
            ax.text(x0, y0 - 4, f'{s:.2f}', color='red', fontsize=8)
    ax.set_title(os.path.basename(test_items[i][0]), fontsize=8); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{RES_DIR}/plane_detections_test.png', dpi=120); plt.show()
print('Saved figures to', RES_DIR, '(download them for the GitHub results/ folder)')